## Dados do Brasileirão de pontos corridos 2007 à 2025

Neste estudo de caso realizamos uma análise exploratória e estatística dos dados históricos do Campeonato Brasileiro para investigar fatores que influenciam o sucesso das equipes. O projeto tem três dashboards:

1. **Desempenho do Campeão** — análise da trajetória dos campeões ao longo do campeonato
2. **Investimento vs Títulos** — correlação valor de mercado × desempenho final, com inferência estatística
3. **Tese da 8ª Rodada** — testar se um bom início de campeonato correlaciona com o título, com correlação em 3 períodos (população completa, 2007-2017, 2018-2025)

## ETL base

Nesta sessão vou fazer a etl para ter um df base

### Instalação de pacotes

In [ ]:
!pip install -q missingno
#com ele conseguimos criar um mapa de valores ausentes 

In [ ]:
!pip install -q category_encoders
#permite categorizar as variaveis

In [ ]:
!pip install -q plotly
#este pacote cria graficos 

In [ ]:
!pip install matplotlib

In [ ]:
# importando bibliotecas para o projeto
# Imports

# Manipulação de dados
import pandas as pd
import numpy as np

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import missingno

# Estatística
import scipy
from scipy.stats import normaltest
from scipy.stats import chi2_contingency

# Engenharia de Atributos
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
import category_encoders as ce

# Ignore Warning
import sys
import warnings
if not sys.warnoptions:
    warnings.simplefilter("ignore")

In [ ]:
!pip install -q -U watermark

In [ ]:
%reload_ext watermark
%watermark -a "Renato"

In [ ]:
# Configuração para os gráficos aparecerem bem no Jupyter
%matplotlib inline
sns.set_theme(style="whitegrid")

In [ ]:
#carregando os dados 
df = pd.read_csv('../dados/raw/Campeonato.csv')

In [ ]:
df.head()

In [ ]:
#como o ano em que o campeonato foi composto por 20 times foi 2006, vou retirar os anos anteriores, para que fique uma análise mais justa 
df = df[df['ano_campeonato'] >= 2007].copy()

In [ ]:
df.head()

In [ ]:
# "vendo" o corpo do df após o filtro
df.shape

In [ ]:
#conferindo as colunas 
df.columns

### Análise exploratória

  Aqui vou saber como o interpretador "classificou" as colunas 

In [ ]:
df.info()

In [ ]:
df.tail()

In [ ]:
# verificando as colunas vazias
df.isna().sum().sort_values(ascending=False)

In [ ]:
# excluindo as com poucos valores ou totalmente vazias
df = df.drop(columns=[
    'impedimentos_mandante',
    'impedimentos_visitante',
    'defesas_mandante',
    'chutes_mandante',
    'chutes_visitante',
    'chutes_fora_mandante'
])

In [ ]:
df.head()

## Desempenho do Campeão

In [ ]:
df.info()

In [ ]:
# Ajustando o recorte para 2007 devido à ausência de dados financeiros em 2006 e criando a coluna temporada devido ao ano de 2020 em que 
# o campeonato terminou no início de 2021
df['data'] = pd.to_datetime(df['data'])
df['temporada'] = df['data'].dt.year
mask_2020_prolongado = (df['data'] >= '2021-01-01') & (df['data'] <= '2021-02-25')
df.loc[mask_2020_prolongado, 'temporada'] = 2020

df = df[df['temporada'] >= 2007].copy()

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
#criando uma lógica para calcular os pontos 
df['pts_mandante'] = np.where(
    df['gols_mandante'] > df['gols_visitante'], 3,
    np.where(df['gols_mandante'] == df['gols_visitante'], 1, 0)
)

df['pts_visitante'] = np.where(
    df['gols_visitante'] > df['gols_mandante'], 3,
    np.where(df['gols_visitante'] == df['gols_mandante'], 1, 0)
)

In [ ]:
# Como vou usar no bi vou tratar esse df criando uma cópia como tabela fato ###

cols_base =[
    'data', 'rodada', 'temporada',
    'time_mandante', 'time_visitante',
    'gols_mandante', 'gols_visitante',
    'pts_mandante', 'pts_visitante'
]
df_base = df[cols_base].copy() 

In [ ]:
df_base.head()

In [ ]:
  # renomeando as colunas 
df_mandante = df_base.rename(columns={
    'time_mandante': 'time',
    'time_visitante': 'adversario',
    'gols_mandante': 'gols_pro',
    'gols_visitante': 'gols_contra',
    'pts_mandante': 'pontos'
})[
    ['data', 'rodada', 'temporada', 'time', 'adversario', 'gols_pro', 'gols_contra', 'pontos']
].copy()

df_mandante['casa_fora'] = 'Casa'

In [ ]:
df_visitante = df_base.rename(columns={
    'time_visitante': 'time',
    'time_mandante': 'adversario',
    'gols_visitante': 'gols_pro',
    'gols_mandante': 'gols_contra',
    'pts_visitante': 'pontos'
})[
    ['data', 'rodada', 'temporada', 'time', 'adversario', 'gols_pro', 'gols_contra', 'pontos']
].copy()

df_visitante['casa_fora'] = 'Fora'

In [ ]:
#concatenando as tabelas 
df_fato = pd.concat([df_mandante, df_visitante], ignore_index=True)


In [ ]:
df_fato.head()

In [ ]:
# verificando valores ausentes nas colunas
df_fato.isna().sum()

In [ ]:
# Preenchendo valor nulo, como a mostra é somente de 2 linhas com valores nulos vou substituir por 0
df_fato[['gols_pro', 'gols_contra']] = df_fato[['gols_pro', 'gols_contra']].fillna(0)

In [ ]:
#criando a coluna saldo
df_fato['saldo'] = df_fato['gols_pro'] - df_fato['gols_contra']

In [ ]:
# um outro problema é que as colunas gols_pro e gols_contra estão como float, e pode gerar problemas nos cálculos de saldo por exemplo,
# vou converter para int e assim criar uma coluna de saldo que funcione no bi

df_fato['gols_pro']= df_fato['gols_pro'].astype(int)
df_fato['gols_contra']= df_fato['gols_contra'].astype(int)
df_fato['saldo']= df_fato['saldo'].astype(int)

In [ ]:
# criando as métricas dos pontos 


df_fato['vitoria'] = (df_fato['pontos'] == 3).astype(int)
df_fato['empate'] = (df_fato['pontos'] == 1).astype(int)
df_fato['derrota'] = (df_fato['pontos'] == 0).astype(int)

In [ ]:
df_fato.head()

In [ ]:
# para ficar mais correto vou inverter a ordem das colunas rodada e temporada
df_fato = df_fato.sort_values(['temporada', 'rodada'])

In [ ]:
# calculando os pontos acumulados de cada time
df_fato['pontos_acum'] = df_fato.groupby(['temporada', 'time'])['pontos'].cumsum()
df_fato['vitorias_acum'] = df_fato.groupby(['temporada', 'time']) ['vitoria'].cumsum()
df_fato['saldo_acum'] = df_fato.groupby(['temporada', 'time']) ['saldo'].cumsum()


# agora posso criar a lógica para a posição na rodada(o que vai ser útil no bi)
df_fato['posicao_rodada'] = (
    df_fato
    .sort_values(
        ['temporada', 'rodada', 'pontos_acum', 'vitorias_acum', 'saldo_acum'],
        ascending=[True, True, False, False, False]
    )
    .groupby(['temporada', 'rodada'])
    .cumcount() + 1
)

In [ ]:
df_fato.head()

In [ ]:
df_fato.columns

### Inserindo dados de outra fonte

  O dataset usado não está completo, não possui as últimas rodadas da temporada de 2024 e a temporada de 2025

In [ ]:
# Lendo os novos arquivos 
df_gols = pd.read_csv('../dados/raw/campeonato-brasileiro-gols.csv')
df_full = pd.read_csv('../dados/raw/campeonato-brasileiro-full.csv')

In [ ]:
#padronizando a nomenclatura das colunas 
df_gols.columns = df_gols.columns.str.lower().str.strip()
df_full.columns = df_full.columns.str.lower().str.strip()

In [ ]:
#renomeando as colunas para padronizar 
df_gols = df_gols.rename(columns={'rodata': 'rodada'})
df_full = df_full.rename(columns={'rodata': 'rodada'})

In [ ]:
df_gols_count = df_gols.groupby(['partida_id', 'clube']).size().reset_index(name='gols')

In [ ]:
#criando a regra para mandante 
mandante = df_full.merge(
    df_gols_count,
    left_on=['id', 'mandante'],
    right_on=['partida_id', 'clube'],
    how='left'
)

mandante['gols_pro'] = mandante['gols'].fillna(0)

In [ ]:
#regra para o visitante
visitante = df_full.merge(
    df_gols_count,
    left_on=['id', 'visitante'],
    right_on=['partida_id', 'clube'],
    how='left'
)

visitante['gols_pro'] = visitante['gols'].fillna(0)

In [ ]:
#convertendo colunas 
df_full['gols_mandante'] = mandante['gols_pro']
df_full['gols_visitante'] = visitante['gols_pro']

In [ ]:
#criando a tabela de mandante 
mandante_df = df_full[['data','rodada','mandante','visitante','gols_mandante','gols_visitante']].copy()

mandante_df.columns = ['data','rodada','time','adversario','gols_pro','gols_contra']
mandante_df['casa_fora'] = 'casa'

In [ ]:
#criando a tabela de visitante 
visitante_df = df_full[['data','rodada','visitante','mandante','gols_visitante','gols_mandante']].copy()

visitante_df.columns = ['data','rodada','time','adversario','gols_pro','gols_contra']
visitante_df['casa_fora'] = 'fora'

In [ ]:
#concatenando as tabelas
df_novo = pd.concat([mandante_df, visitante_df], ignore_index=True)

In [ ]:
#convertendo as colunas para o mesmo type da df_fato para evitar granularidade
df_novo['pontos'] = np.where(df_novo['gols_pro'] > df_novo['gols_contra'], 3,
                     np.where(df_novo['gols_pro'] == df_novo['gols_contra'], 1, 0))

df_novo['vitoria'] = (df_novo['pontos'] == 3).astype(int)
df_novo['empate'] = (df_novo['pontos'] == 1).astype(int)
df_novo['derrota'] = (df_novo['pontos'] == 0).astype(int)

df_novo['saldo'] = df_novo['gols_pro'] - df_novo['gols_contra']

In [ ]:
#Criando a coluna temporada 
df_novo['temporada'] = pd.to_datetime(df_novo['data']).dt.year

In [ ]:
#criando as colunas de pontos, vitorias e saldo acumulados
df_novo = df_novo.sort_values(['temporada','time','rodada'])

df_novo['pontos_acum'] = df_novo.groupby(['temporada','time'])['pontos'].cumsum()
df_novo['vitorias_acum'] = df_novo.groupby(['temporada','time'])['vitoria'].cumsum()
df_novo['saldo_acum'] = df_novo.groupby(['temporada','time'])['saldo'].cumsum()

In [ ]:
#criando a coluna de posição na rodada
df_novo['posicao_rodada'] = df_novo.groupby(['temporada','rodada'])['pontos_acum'] \
    .rank(ascending=False, method='min')

In [ ]:
#vendo se o df ficou correto
df_novo.head()

In [ ]:
#pelo head vimos que 3 colunas ainda continuam como float, convertendo para int
cols_int = ['gols_pro','gols_contra','saldo','posicao_rodada']

df_novo[cols_int] = df_novo[cols_int].astype(int)

In [ ]:
df_novo.tail()

In [ ]:
#verificando se ambos os arquivos possuem as mesmas colunas 
set(df_novo.columns) == set(df_fato.columns)

In [ ]:
#concatenando os df 
df_total = pd.concat([df_fato, df_novo], ignore_index=True)

In [ ]:
df_total.head()

In [ ]:
df_total.tail()

In [ ]:
#agora a temporada de 2025 está sendo mostrada 
# Garantindo que não haja duplicatas antes de salvar
df_total = df_total.drop_duplicates()

In [ ]:
# 1. Padronizar para minúsculo para evitar "Casa" vs "casa"
df_total['casa_fora'] = df_total['casa_fora'].str.lower()

# 2. Remover duplicatas considerando a chave: temporada, rodada e time
# O 'keep=first' mantém apenas a primeira ocorrência de cada jogo, estavadando problema no bi
df_total = df_total.drop_duplicates(subset=['temporada', 'rodada', 'time'], keep='first')

# 3. Filtrar a partir de 2007 
df_total = df_total[df_total['temporada'] >= 2007]

In [ ]:
df_total.to_csv('../dados/processed/fato_desempenho2.csv', index=False)

## Ivenstimento vs Desempenho

In [ ]:
df = pd.read_csv('../dados/raw/Campeonato.csv')

In [ ]:
df = df[df['ano_campeonato'] >= 2007].copy()

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
# excluindo as com poucos valores ou totalmente vazias

df = df.drop(columns=[
    'impedimentos_mandante',
    'impedimentos_visitante',
    'defesas_mandante',
    'chutes_mandante',
    'chutes_visitante',
    'chutes_fora_mandante',
    'escanteios_mandante',
    'escanteios_visitante', # Corrigido: removido o 's'
    'faltas_mandante',
    'faltas_visitante',
    'chutes_bola_parada_mandante', # Corrigido: adicionada a vírgula
    'chutes_bola_parada_visitante'
])

In [ ]:
# Analisando a cobertura de dados financeiros por ano
financeiro_por_ano = df.groupby('ano_campeonato').apply(lambda x: x['valor_equipe_titular_mandante'].notnull().mean() * 100)
print("Porcentagem de dados financeiros presentes por ano:")
print(financeiro_por_ano)

# Plotando para visualização rápida
plt.figure(figsize=(10,4))
financeiro_por_ano.plot(kind='bar', color='skyblue')
plt.title('% de Jogos com Valor de Mercado Disponível')
plt.ylabel('% Preenchido')
plt.show()

In [ ]:
df.columns

In [ ]:
df.to_csv('../dados/raw/Investimento.csv')

In [ ]:
pd.options.display.float_format = '{:.2f}'.format

In [ ]:
df_invest = pd.read_csv('../dados/raw/Investimento.csv')

In [ ]:
df_invest.head()

In [ ]:
df_invest.columns

In [ ]:
df_mandante = df_invest[['ano_campeonato','rodada','time_mandante','valor_equipe_titular_mandante']].rename(columns={
    'time_mandante': 'time',
    'valor_equipe_titular_mandante': 'valor_mercado'
})

df_visitante = df_invest[['ano_campeonato','rodada','time_visitante','valor_equipe_titular_visitante']].rename(columns={
    'time_visitante': 'time',
    'valor_equipe_titular_visitante': 'valor_mercado'
})

df_valor = pd.concat([df_mandante, df_visitante])

In [ ]:
df_valor['temporada'] = df_valor['ano_campeonato']

In [ ]:
df_valor_time = df_valor.groupby(['temporada','time']).agg({
    'valor_mercado': 'mean' 
}).reset_index()

In [ ]:
df_valor_time.head()

In [ ]:
# pelo head, logo na primeira linha pode se observar um valor nulo, vou ver quantos são ao total
df_valor_time['valor_mercado'].isna().sum()

In [ ]:
# Filtra as linhas onde a coluna 'valor_mercado' é nula (NaN), e com o print vou poder saber quais os times 
nulos = df_valor_time[df_valor_time['valor_mercado'].isna()]

print(nulos)

Observou-se a ausência de um time na temporada de 2007 (América de Natal),
último colocado com baixo investimento.

Considerando seu baixo impacto no desempenho geral e no padrão da relação
entre investimento e pontuação, optou-se por manter o dataset sem ajustes,
pois não compromete a análise estatística.


Não foi possível obter o valor de mercado do Figueirense  na temporada de 2011.
Optou-se por remover o registro para garantir a integridade da análise de correlação,
evitando a introdução de dados artificiais.

In [ ]:
df_modelo = df_fato.groupby(['temporada','time']).agg({
    'pontos': 'sum',
    'saldo': 'sum',
    'vitoria': 'sum'
}).reset_index()

df_modelo = df_modelo.merge(df_valor_time, on=['temporada','time'], how='left')

In [ ]:
df_modelo.head()

In [ ]:
df_modelo['valor_mercado'].isna().sum()

In [ ]:
df_modelo.head()

In [ ]:
df_modelo = df_modelo.dropna(subset=['valor_mercado'])

In [ ]:
df_modelo.head()

In [ ]:
df_modelo.tail()

In [ ]:
# ==========================================================
# INTEGRAÇÃO DA TEMPORADA 2025 (dados de outraFonte / fato_desempenho.csv)
# ==========================================================
# Os valores de mercado de 2025 foram obtidos da ListaDeTimes.docx
val_2025 = {
    'Palmeiras': 238_750_000, 'Flamengo': 219_150_000,
    'Botafogo-RJ': 135_950_000, 'Corinthians': 123_250_000,
    'Internacional': 106_650_000, 'Gremio': 98_100_000,
    'Santos': 96_100_000, 'Atletico-MG': 93_700_000,
    'Bahia': 92_000_000, 'Cruzeiro': 89_550_000,
    'Vasco': 85_880_000, 'Sao Paulo': 80_650_000,
    'Fluminense': 73_600_000, 'Bragantino': 70_100_000,
    'Fortaleza': 47_850_000, 'Vitoria': 39_950_000,
    'Sport': 36_650_000, 'Ceara': 30_000_000,
    'Mirassol': 18_230_000, 'Juventude': 17_380_000,
}

# Verificar se 2025 está presente no fato
fato_all = pd.read_csv('../dados/processed/fato_desempenho.csv', encoding='latin1')
fato_all = fato_all[fato_all['temporada'] >= 2007].copy()
tem_2025 = 2025 in fato_all['temporada'].values

if tem_2025:
    print("2025 encontrado no fato_desempenho.csv — integrando...")
    
    fato_2025 = fato_all[fato_all['temporada'] == 2025].copy()
    
    # Reconstruir jogos de 2025 a partir da tabela fato
    jogos_2025 = []
    for rod in sorted(fato_2025['rodada'].unique()):
        rd = fato_2025[fato_2025['rodada'] == rod]
        casa = rd[rd['casa_fora'] == 'casa'].reset_index(drop=True)
        fora = rd[rd['casa_fora'] == 'fora'].reset_index(drop=True)
        for i in range(len(casa)):
            man = casa.loc[i, 'time']
            vis = casa.loc[i, 'adversario']
            away_row = fora[fora['time'] == vis]
            gols_vis = float(away_row.iloc[0]['gols_pro']) if len(away_row) > 0 else np.nan
            jogos_2025.append({
                'ano_campeonato': 2025, 'rodada': rod, 'temporada': 2025,
                'time_mandante': man, 'time_visitante': vis,
                'gols_mandante': float(casa.loc[i, 'gols_pro']),
                'gols_visitante': gols_vis,
                'valor_equipe_titular_mandante': val_2025.get(man, np.nan),
                'valor_equipe_titular_visitante': val_2025.get(vis, np.nan),
            })
    
    df_2025_inv = pd.DataFrame(jogos_2025)
    
    # Unir valores: fontes originais (2007-2024) + 2025
    vals_25_man = df_2025_inv[['temporada', 'time_mandante', 'valor_equipe_titular_mandante']].rename(
        columns={'time_mandante': 'time', 'valor_equipe_titular_mandante': 'valor_mercado'})
    vals_25_vis = df_2025_inv[['temporada', 'time_visitante', 'valor_equipe_titular_visitante']].rename(
        columns={'time_visitante': 'time', 'valor_equipe_titular_visitante': 'valor_mercado'})
    df_valor_25 = pd.concat([vals_25_man, vals_25_vis], ignore_index=True)
    
    # Recalcular df_valor_time com 2007-2025
    df_valor_all = pd.concat([df_valor, df_valor_25], ignore_index=True)
    df_valor_time = df_valor_all.groupby(['temporada', 'time']).agg(
        valor_mercado=('valor_mercado', 'mean')
    ).reset_index().dropna(subset=['valor_mercado'])
else:
    print("2025 NAO encontrado no fato_desempenho.csv. Usando dados 2007-2024.")
    print("Execute outraFonte.ipynb para adicionar 2025 e rode este notebook novamente.")
    df_valor_all = df_valor.copy()
    df_valor_time = df_valor_all.groupby(['temporada', 'time']).agg(
        valor_mercado=('valor_mercado', 'mean')
    ).reset_index().dropna(subset=['valor_mercado'])

# Recalcular df_modelo
df_perf = fato_all[fato_all['temporada'] != 2025].groupby(['temporada', 'time']).agg(
    pontos=('pontos', 'sum'),
    saldo=('saldo', 'sum'),
    vitoria=('vitoria', 'sum')
).reset_index()

if tem_2025:
    df_perf_25 = fato_all[fato_all['temporada'] == 2025].groupby(['temporada', 'time']).agg(
        pontos=('pontos', 'sum'),
        saldo=('saldo', 'sum'),
        vitoria=('vitoria', 'sum')
    ).reset_index()
    df_perf = pd.concat([df_perf, df_perf_25], ignore_index=True)

df_modelo = df_perf.merge(df_valor_time, on=['temporada', 'time'], how='left')
df_modelo = df_modelo.dropna(subset=['valor_mercado'])

# Rankings
df_modelo['rank_performance'] = df_modelo.groupby('temporada')['pontos'].rank(method='min', ascending=False)
df_modelo['rank_financeiro'] = df_modelo.groupby('temporada')['valor_mercado'].rank(method='min', ascending=False)
df_modelo['no_g4_real'] = (df_modelo['rank_performance'] <= 4).astype(int)
df_modelo['no_g4_financeiro'] = (df_modelo['rank_financeiro'] <= 4).astype(int)
df_modelo['destaque_gestao'] = df_modelo.apply(lambda r: 'Duo Dominante (Fla/Pal)' if r['temporada'] >= 2018 and r['time'] in ['Flamengo', 'Palmeiras'] else 'Outros', axis=1)

print(f"df_modelo: {df_modelo.shape}")
print(f"Temporadas: {sorted(df_modelo['temporada'].unique())}")
print(df_modelo.groupby('temporada').size())

In [ ]:
df_modelo.head()

In [ ]:
df_modelo.tail()

### Evento de ruptura
   Antes de 2018, o Brasileirão era conhecido pela sua extrema alternância (o "qualquer um ganha"). A organização desses dois clubes mudou o comportamento da  variável pontos em relação ao valor_mercado: o dinheiro passou a ser convertido em pontos de forma muito mais eficiente e previsível.

In [ ]:
#fazendo a correlação de pearson sobre o valor de mercado 
from scipy.stats import pearsonr

corr, p_value = pearsonr(df_modelo['valor_mercado'], df_modelo['pontos'])

print(corr, p_value)

In [ ]:
# 1. Definir os rankings por temporada
df_modelo['rank_performance'] = df_modelo.groupby('temporada')['pontos'].rank(method='min', ascending=False)
df_modelo['rank_financeiro'] = df_modelo.groupby('temporada')['valor_mercado'].rank(method='min', ascending=False)

# 2. Criar colunas binárias (1 se for Top 4, 0 se não)
df_modelo['no_g4_real'] = (df_modelo['rank_performance'] <= 4).astype(int)
df_modelo['no_g4_financeiro'] = (df_modelo['rank_financeiro'] <= 4).astype(int)

# 3. Calcular a Matrix de Confusão / Eficiência
# Quantas vezes o Top 4 financeiro REALMENTE terminou no G4?
sucesso_investimento = df_modelo[df_modelo['no_g4_financeiro'] == 1]['no_g4_real'].mean() * 100

print(f"Eficiência do Capital: {sucesso_investimento:.1f}% dos times do Top 4 financeiro garantiram vaga na Libertadores.")

In [ ]:
# 1. Definir o ranking de pontos (posicao final) e o ranking financeiro por temporada
# Usamos o método 'rank' com 'ascending=False' para que o maior valor seja 1º
df_modelo['rank_final'] = df_modelo.groupby('temporada')[['pontos', 'vitoria', 'saldo']].apply(
    lambda x: x.sort_values(['pontos', 'vitoria', 'saldo'], ascending=False)
).index.get_level_values(1) # Este passo é complexo, vamos simplificar:

# Forma mais simples de garantir os rankings:
df_modelo['rank_performance'] = df_modelo.groupby('temporada')['pontos'].rank(method='min', ascending=False)
df_modelo['rank_financeiro'] = df_modelo.groupby('temporada')['valor_mercado'].rank(method='min', ascending=False)

# 2. Filtrar apenas os campeões (rank_performance == 1)
# Obs: Em caso de empate exato em tudo, pegamos o primeiro
campeoes = df_modelo.sort_values(['temporada', 'pontos', 'vitoria', 'saldo'], ascending=[True, False, False, False])
campeoes = campeoes.groupby('temporada').head(1)

# 3. Preparar o gráfico do ranking financeiro dos campeões
plt.figure(figsize=(12, 6))
sns.barplot(data=campeoes, x='temporada', y='rank_financeiro', palette='viridis')

plt.axhline(y=4, color='red', linestyle='--', label='Top 4 Financeiro')
plt.title('Ranking Financeiro do Campeão (Qual era a posição do campeão em investimento?)')
plt.ylabel('Posição no Ranking de Valor de Mercado')
plt.xlabel('Temporada')
plt.gca().invert_yaxis() # Inverter para o 1º ficar no topo
plt.yticks(range(1, int(campeoes['rank_financeiro'].max()) + 1))
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.regplot(data=df_modelo, x='valor_mercado', y='pontos', 
            scatter_kws={'alpha':0.5, 'color':'gray'}, 
            line_kws={'color':'red', 'label':'Tendência'})

# Destacar quem ficou no G4
g4_plot = df_modelo[df_modelo['no_g4_real'] == 1]
plt.scatter(g4_plot['valor_mercado'], g4_plot['pontos'], color='blue', label='Times no G4')

plt.title('Correlação: Valor de Mercado vs. Pontos Totais (Destaque G4)')
plt.xlabel('Investimento (Valor de Mercado)')
plt.ylabel('Pontuação Final')
plt.legend()
plt.show()

In [ ]:
# Boa gestão e seus resultados 
def destacar_dominancia(row):
    # Filtro para o período de hegemonia e os clubes específicos
    if row['temporada'] >= 2018 and row['time'] in ['Flamengo', 'Palmeiras']:
        return 'Duo Dominante (Fla/Pal)'
    return 'Outros'

# 2. Aplicando a função corrigida (destacar_dominancia)
df_modelo['destaque_gestao'] = df_modelo.apply(destacar_dominancia, axis=1)

# 3. Gerando o gráfico para visualizar o "Salto de Patamar"
plt.figure(figsize=(10,6))
sns.scatterplot(data=df_modelo, x='valor_mercado', y='pontos', 
                hue='destaque_gestao', style='destaque_gestao', 
                palette={'Duo Dominante (Fla/Pal)': 'gold', 'Outros': 'gray'},
                s=100, alpha=0.8)

plt.title('Impacto da Gestão: Flamengo e Palmeiras vs Restante (Pós-2018)')
plt.xlabel('Valor de Mercado')
plt.ylabel('Pontos Totais')
plt.legend(title='Categoria')
plt.show()

In [ ]:
# ==========================================================
# CORRELAÇÃO PEARSON EM 3 PERÍODOS
# ==========================================================

# FASE 1: População completa (2007-2025)
corr_tot, p_tot = pearsonr(df_modelo['valor_mercado'], df_modelo['pontos'])
print(f'=== FASE 1: População completa (2007-2025) ===')
print(f'Pearson r = {corr_tot:.4f}')
print(f'p-value   = {p_tot:.2e}')
print(f'n         = {len(df_modelo)}')
print()

# FASE 2: Pré-ruptura (2007-2017)
df_pre = df_modelo[df_modelo['temporada'] <= 2017]
corr_pre, p_pre = pearsonr(df_pre['valor_mercado'], df_pre['pontos'])
print(f'=== FASE 2: Pré-ruptura (2007-2017) ===')
print(f'Pearson r = {corr_pre:.4f}')
print(f'p-value   = {p_pre:.2e}')
print(f'n         = {len(df_pre)}')
print()

# FASE 3: Pós-ruptura (2018-2025)
df_pos = df_modelo[df_modelo['temporada'] >= 2018]
corr_pos, p_pos = pearsonr(df_pos['valor_mercado'], df_pos['pontos'])
print(f'=== FASE 3: Pós-ruptura (2018-2025) ===')
print(f'Pearson r = {corr_pos:.4f}')
print(f'p-value   = {p_pos:.2e}')
print(f'n         = {len(df_pos)}')

In [ ]:
# ==========================================================
# TESTES DE HIPÓTESE  —  Investimento vs Pontos
# ==========================================================
from scipy.stats import norm

print("=== TESTE 1: r > 0 em cada período (Fisher z one-tail) ===")
for label, r, p, n in [
    ('Completo', corr_tot, p_tot, len(df_modelo)),
    ('Pré-ruptura', corr_pre, p_pre, len(df_pre)),
    ('Pós-ruptura', corr_pos, p_pos, len(df_pos)),
]:
    z = np.arctanh(r)
    se = 1 / np.sqrt(n - 3)
    z_stat = z / se
    p1 = 1 - norm.cdf(z_stat)  # one-tail
    sig = 'SIM' if p1 < 0.05 else 'NÃO'
    print(f'{label}: r={r:.4f}, z={z_stat:.3f}, p={p1:.2e} → Rejeita H0? {sig}')

print()
print("=== TESTE 2: r_pós > r_pré? (Fisher z-test para diferença entre correlações) ===")
z_pre = np.arctanh(corr_pre)
z_pos = np.arctanh(corr_pos)
se_diff = np.sqrt(1/(len(df_pre) - 3) + 1/(len(df_pos) - 3))
z_diff = (z_pos - z_pre) / se_diff
p_diff = 1 - norm.cdf(z_diff)
print(f'z_pre = {z_pre:.4f} (r={corr_pre:.4f})')
print(f'z_pos = {z_pos:.4f} (r={corr_pos:.4f})')
print(f'z_diff = {z_diff:.4f}, p_one-tail = {p_diff:.4f}')
print(f'A correlação aumentou significativamente após 2018? {"SIM" if p_diff < 0.05 else "NÃO"} (α=0.05)')

In [ ]:
# Teste: Campeões tendem a ser do Top 4 financeiro?
from scipy.stats import ttest_1samp

campeoes = df_modelo.sort_values(['temporada', 'pontos', 'vitoria', 'saldo'],
                                  ascending=[True, False, False, False])
campeoes = campeoes.groupby('temporada').head(1).copy()
campeoes['top4_fin'] = (campeoes['rank_financeiro'] <= 4).astype(int)

pct = campeoes['top4_fin'].mean() * 100
# Teste t one-tail vs proporção aleatória de 50%
t_stat, p_val = ttest_1samp(campeoes['top4_fin'], 0.5)
p_one = p_val / 2 if t_stat > 0 else 1 - p_val / 2

print("=== TESTE 3: Campeões estão no Top 4 financeiro? ===")
print(f"Campeões no Top 4: {campeoes['top4_fin'].sum()}/{len(campeoes)} ({pct:.1f}%)")
print(f't = {t_stat:.3f}, p_one-tail = {p_one:.4f}')
print(f"Significativo? {'SIM' if p_one < 0.05 else 'NÃO'}")

# Binomial test
import scipy.stats as stats
binom_p = 1 - stats.binom.cdf(campeoes['top4_fin'].sum() - 1, len(campeoes), 0.20)
print(f'Teste binomial (H0: 20% aleatório): {binom_p:.4f}')

In [ ]:
# GERAR CSV ANÁLISE DE INVESTIMENTO PARA POWER BI
df_bi = df_modelo[[
    'temporada', 'time', 'pontos', 'saldo', 'vitoria',
    'valor_mercado', 'rank_performance', 'rank_financeiro',
    'no_g4_real', 'no_g4_financeiro', 'destaque_gestao'
]].copy()
df_bi.to_csv('../dados/processed/investimento_vs_desempenho.csv', index=False, sep=';', encoding='utf-8')

print(f'investimento_vs_desempenho.csv salvo: {df_bi.shape}')
print(df_bi.head())
print()
print('Resumo dos campeões:')
print(campeoes[['temporada', 'time', 'pontos', 'rank_financeiro', 'top4_fin']].to_string())

## Tese do Guardiola

  A tese do técnico Pepe Guardiola consiste em que se ganha títulos nas 8 últimas rodadas, mas você perde eles nas 8 primeiras, nesta sessão 
vamos analisar se essa tese se aplica aqui no Brasil.

In [ ]:
df.drop(columns=['colunas_que_talvez_nao_existam'], errors='ignore', inplace=True)

In [ ]:
# ==========================================================
# TESE DA 8ª RODADA (Guardiola)
# ==========================================================
# Função para gerar classificação a partir de um df no formato
# com colunas: time, pontos, gols_pro, gols_contra
def gerar_classificacao_simples(df_periodo):
    tabela = df_periodo.groupby('time').agg(
        pts=('pontos', 'sum'),
        gp=('gols_pro', 'sum'),
        gc=('gols_contra', 'sum')
    )
    tabela['sg'] = tabela['gp'] - tabela['gc']
    return tabela.sort_values(by=['pts', 'sg', 'gp'], ascending=False)

resultados = []

# Para os anos 2007-2024, usar Investment.csv
df_orig = pd.read_csv('../dados/raw/Investimento.csv')
df_orig = df_orig[df_orig['ano_campeonato'] >= 2007].copy()
df_orig['pontos_m'] = np.where(
    df_orig['gols_mandante'] > df_orig['gols_visitante'], 3,
    np.where(df_orig['gols_mandante'] == df_orig['gols_visitante'], 1, 0))
df_orig['pontos_v'] = np.where(
    df_orig['gols_visitante'] > df_orig['gols_mandante'], 3,
    np.where(df_orig['gols_visitante'] == df_orig['gols_mandante'], 1, 0))

for ano in sorted(df_orig['ano_campeonato'].unique()):
    df_ano = df_orig[df_orig['ano_campeonato'] == ano]
    m = df_ano[['rodada','time_mandante','pontos_m','gols_mandante','gols_visitante']].rename(
        columns={'time_mandante': 'time', 'pontos_m': 'pontos', 'gols_mandante': 'gols_pro', 'gols_visitante': 'gols_contra'})
    v = df_ano[['rodada','time_visitante','pontos_v','gols_visitante','gols_mandante']].rename(
        columns={'time_visitante': 'time', 'pontos_v': 'pontos', 'gols_visitante': 'gols_pro', 'gols_mandante': 'gols_contra'})
    df_ano_all = pd.concat([m, v], ignore_index=True)
    
    t_r8 = gerar_classificacao_simples(df_ano_all[df_ano_all['rodada'] <= 8])
    if len(t_r8) == 0:
        continue
    lider_r8 = t_r8.index[0]
    pts_r8 = t_r8.iloc[0]['pts']
    t_fin = gerar_classificacao_simples(df_ano_all)
    campeao_fin = t_fin.index[0]
    pos_fin = t_fin.index.get_loc(lider_r8) + 1
    aprov_r8 = (pts_r8 / 24) * 100
    resultados.append({
        'temporada': ano, 'lider_r8': lider_r8, 'pts_r8': pts_r8,
        'aproveitamento_r8': aprov_r8, 'campeao': campeao_fin,
        'pos_final_lider_r8': pos_fin,
        'tese_confirmada': 1 if lider_r8 == campeao_fin else 0
    })

# Processar 2025 a partir do fato_desempenho.csv
fato_25_r8 = pd.read_csv('../dados/processed/fato_desempenho.csv', encoding='latin1')
fato_25_r8 = fato_25_r8[fato_25_r8['temporada'] == 2025].copy()
if len(fato_25_r8) > 0:
    print("Processando 2025 para a Tese R8...")
    t_r8_25 = gerar_classificacao_simples(fato_25_r8[fato_25_r8['rodada'] <= 8])
    t_fin_25 = gerar_classificacao_simples(fato_25_r8)
    
    if len(t_r8_25) > 0 and len(t_fin_25) > 0:
        lider_r8_25 = t_r8_25.index[0]
        pts_r8_25 = t_r8_25.iloc[0]['pts']
        campeao_25 = t_fin_25.index[0]
        pos_fin_25 = t_fin_25.index.get_loc(lider_r8_25) + 1
        aprov_r8_25 = (pts_r8_25 / 24) * 100
        resultados.append({
            'temporada': 2025, 'lider_r8': lider_r8_25, 'pts_r8': pts_r8_25,
            'aproveitamento_r8': aprov_r8_25, 'campeao': campeao_25,
            'pos_final_lider_r8': pos_fin_25,
            'tese_confirmada': 1 if lider_r8_25 == campeao_25 else 0
        })
        print(f"  2025: Lider R8 = {lider_r8_25} ({pts_r8_25} pts), Campeao = {campeao_25}, Posicao Final = {pos_fin_25}")

df_validacao = pd.DataFrame(resultados)
df_validacao['periodo'] = np.where(
    df_validacao['temporada'] <= 2017, 'Pre-ruptura (2007-2017)',
    'Pos-ruptura (2018-2025)')

# Teste binomial — Líder R8 no Top 4
top4_count = (df_validacao['pos_final_lider_r8'] <= 4).sum()
total = len(df_validacao)
binom_p_top4 = 1 - stats.binom.cdf(top4_count - 1, total, 0.20)

print(f'Teste binomial — Líder R8 no Top 4: {top4_count}/{total} ({top4_count/total*100:.1f}%)')
print(f'p_one-tail (H0: 20% aleatório) = {binom_p_top4:.4f}')
print(f"Significativo? {'SIM' if binom_p_top4 < 0.05 else 'NÃO'}")

# Resumo por período
for periodo, df_v in df_validacao.groupby('periodo'):
    teses = df_v['tese_confirmada'].sum()
    total_p = len(df_v)
    print(f"\n{periodo}: Tese confirmada {teses}/{total_p} ({teses/total_p*100:.1f}%)")

In [ ]:
# Gráfico: Líder R8 - Posição Inicial vs Final por período
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, df_v, titulo in [
    (axes[0], df_validacao[df_validacao['temporada'] <= 2017], 'Pre-ruptura (2007-2017)'),
    (axes[1], df_validacao[df_validacao['temporada'] >= 2018], 'Pos-ruptura (2018-2025)')
]:
    for _, row in df_v.iterrows():
        cor = 'green' if row['tese_confirmada'] == 1 else 'red'
        ax.scatter(['R8', 'Final'], [1, row['pos_final_lider_r8']], color=cor, s=100, zorder=3)
        ax.plot(['R8', 'Final'], [1, row['pos_final_lider_r8']], color=cor, alpha=0.4)
        if row['tese_confirmada'] == 1:
            ax.annotate('Campeao', xy=(0.5, 1.05), ha='center', color='green', fontweight='bold')
        else:
            ax.annotate(f"{row['pos_final_lider_r8']}o", xy=(0.5, 1.05), ha='center', color='red')
    ax.set_ylim(20.5, 0.5)
    ax.set_title(titulo, fontweight='bold')
    ax.grid(True, alpha=0.3)
fig.suptitle('Lider da 8a Rodada - Posicao Inicial vs Posicao Final', fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

# GERAR CSV TESE DA 8ª RODADA PARA POWER BI
df_r8_bi = df_validacao[[
    'temporada', 'lider_r8', 'pts_r8', 'aproveitamento_r8',
    'campeao', 'pos_final_lider_r8', 'tese_confirmada', 'periodo'
]].copy()
df_r8_bi.to_csv('../dados/processed/tese_r8_guardiola.csv', index=False, sep=';', encoding='utf-8')
print(f'\ntese_r8_guardiola.csv salvo: {df_r8_bi.shape}')
print(df_r8_bi.to_string())

In [ ]:
# ==========================================================
# INTERVALOS DE CONFIANÇA — Dashboard 3 (Tese do Guardiola)
# ==========================================================
# Recarrega o CSV salvo e adiciona colunas de IC (95%)
# Os valores são repetidos em todas as linhas para uso como
# cartões no Power BI via medidas MAX/MIN.
import numpy as np
from scipy import stats
import pandas as pd

df_ic = pd.read_csv('../dados/processed/tese_r8_guardiola.csv', sep=';', encoding='utf-8')
n = len(df_ic)
alpha = 0.05

# --- IC 1: Proporção (Wilson score) — taxa de confirmação da tese ---
p_hat = df_ic['tese_confirmada'].mean()
z = stats.norm.ppf(1 - alpha / 2)
denom = 1 + z**2 / n
center = (p_hat + z**2 / (2 * n)) / denom
margin = (z * np.sqrt(p_hat * (1 - p_hat) / n + z**2 / (4 * n**2))) / denom
ic_prop_lower = round((center - margin) * 100, 1)
ic_prop_upper = round((center + margin) * 100, 1)
print(f'IC Proporção (Wilson 95%): [{ic_prop_lower}%, {ic_prop_upper}%]')

# --- IC 2: Correlação de Pearson (Fisher z-transform) ---
# aproveitamento_r8 vs pos_final_lider_r8
r, _ = stats.pearsonr(df_ic['aproveitamento_r8'], df_ic['pos_final_lider_r8'])
z_r = np.arctanh(r)
se_r = 1 / np.sqrt(n - 3)
ic_r_lower = round(np.tanh(z_r - z * se_r), 4)
ic_r_upper = round(np.tanh(z_r + z * se_r), 4)
print(f'IC Correlação Pearson (Fisher 95%): r={r:.4f}, IC=[{ic_r_lower}, {ic_r_upper}]')

# --- IC 3: Posição final média do líder da R8 (t de Student) ---
pos_mean = df_ic['pos_final_lider_r8'].mean()
pos_se = df_ic['pos_final_lider_r8'].sem()
t_crit = stats.t.ppf(1 - alpha / 2, df=n - 1)
ic_pos_lower = round(pos_mean - t_crit * pos_se, 2)
ic_pos_upper = round(pos_mean + t_crit * pos_se, 2)
print(f'IC Posição Final Média (t 95%): média={pos_mean:.2f}, IC=[{ic_pos_lower}, {ic_pos_upper}]')

# --- Adiciona colunas ao CSV ---
df_ic['ic_prop_lower']  = ic_prop_lower
df_ic['ic_prop_upper']  = ic_prop_upper
df_ic['ic_r_lower']     = ic_r_lower
df_ic['ic_r_upper']     = ic_r_upper
df_ic['ic_pos_lower']   = ic_pos_lower
df_ic['ic_pos_upper']   = ic_pos_upper

df_ic.to_csv('../dados/processed/tese_r8_guardiola.csv', index=False, sep=';', encoding='utf-8')
print(f'
tese_r8_guardiola.csv atualizado com ICs: {df_ic.shape}')
